In [1]:

!git clone https://github.com/espressif/esp-ppq.git
%cd esp-ppq


!sed -i 's/python_version < "3.12"/python_version < "4.0"/g' pyproject.toml


!pip install .


fatal: destination path 'esp-ppq' already exists and is not an empty directory.
/content/esp-ppq
Processing /content/esp-ppq
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached onnx-1.17.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (16 kB)
Using cached onnx-1.17.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.0 MB)
  Created wheel for esp-ppq: filename=esp_ppq-1.2.9-py3-none-any.whl size=623102 sha256=3eb0e9c450d2e3796b77454b4f3822cde83c8dda08481b9b1317fff0939bdf0d
  Stored in directory: /root/.cache/pip/wheels/81/56/dc/c5d15a0e680f912d5b852f4befedceb53f2e21375e93597d93
Successfully built esp-ppq
  Attempting uninstall: onnx
    Found existing installation: onnx 1.21.0
    Uninstalling onnx-1.21.0:
      Successfully uninstalled onnx-1.21.0
  Attempting uninstall: esp-ppq
    Found existing installation: esp-ppq 1.2.9
    Uninstalling esp-ppq-1.2.9

In [ ]:
!pip install -U tf2onnx onnx onnxruntime
!pip install onnxsim==0.4.24


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 21.1 MB/s eta 0:00:00
  Attempting uninstall: onnx
    Found existing installation: onnx 1.17.0
    Uninstalling onnx-1.17.0:
      Successfully uninstalled onnx-1.17.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
esp-ppq 1.2.9 requires onnx<1.18.0,>=1.14.0, but you have onnx 1.21.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 14.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import esp_ppq


In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# --- 1. Carregar Cats vs Dogs com split train/test ---
(train_data, test_data), info = tfds.load(
    "cats_vs_dogs",
    as_supervised=True,
    with_info=True,
    split=["train[:80%]", "train[80%:]"]
)

img_height, img_width = 128, 128  # shape do modelo ONNX
num_calib_images = 200  # número de imagens para calibração

# --- 2. Função de pré-processamento: redimensiona e normaliza ---
def preprocess(image, label):
    image = tf.image.resize(image, [img_height, img_width])
    image = tf.image.convert_image_dtype(image, tf.float32)  # normaliza para [0,1]
    image = image[..., :3]  # garante 3 canais (RGB)
    return image

# --- 3. Aplicar preprocessing e coletar imagens em numpy ---
images_list = []
for i, (img, lbl) in enumerate(train_data.take(num_calib_images)):
    img = preprocess(img, lbl)
    images_list.append(img.numpy())

images_np = np.stack(images_list, axis=0)  # shape [N, H, W, C] = [200, 224, 224, 3]

# --- 4. Converter para PyTorch tensor mantendo [N, H, W, C] ---
images_pt = torch.tensor(images_np).float()  # shape [N, 224, 224, 3]

# --- 5. Criar TensorDataset e DataLoader ---
calib_dataset = TensorDataset(images_pt)
calib_loader = DataLoader(calib_dataset, batch_size=1, shuffle=True)

# --- 6. Conferir shapes ---
for batch in calib_loader:
    print("Batch shape:", batch[0].shape)
    break

print("Calib loader criado com", len(calib_loader), "batches")


In [ ]:
!pip install tf2onnx


In [ ]:
import tf2onnx

# 1. Caminho para o modelo TFLite
tflite_path = "/content/MobileNetV2_seed_3_lr_0.01_bs16.tflite"

# 2. Converter para ONNX
onnx_model, _ = tf2onnx.convert.from_tflite(
    tflite_path,
    opset=13,
    output_path="/content/base_from_tflite.onnx"
)

print("✅ Conversão completa: salvo como base_from_tflite.onnx")

In [ ]:
from esp_ppq.api import espdl_quantize_onnx
from esp_ppq.executor import TorchExecutor

# Quantização do modelo ONNX
quanted_graph = espdl_quantize_onnx(
    onnx_import_file="/content/base_from_tflite.onnx",       # modelo de entrada
    espdl_export_file="model.espdl",     # saída do modelo quantizado no formato ESP-DL
    calib_dataloader=calib_loader,       # dataloader com dados de calibração
    calib_steps=20,                     # número de batches de calibração
    input_shape=[1,128, 128, 3],        # substitua pelo shape real do seu modelo
    inputs=None,
    target="esp32",
    num_of_bits=8,                       # quantização INT8
    device="cpu"
)



In [ ]:
import torch
import tensorflow as tf
import tensorflow_datasets as tfds

# Suposição: você já tem a variável 'quanted_graph' carregada
# e a classe 'TorchExecutor' definida.
# from sua_lib import TorchExecutor, quanted_graph

# --- 1. Pegar 100 imagens do dataset ---
# Aumentei para 100 para ter uma avaliação mais robusta
dataset = tfds.load("cats_vs_dogs", split="train[80%:]", as_supervised=True).take(100)

img_height, img_width = 128, 128

def preprocess_for_inference(image):
    image = tf.image.resize(image, [img_height, img_width])
    image = tf.image.convert_image_dtype(image, tf.float32)  # normaliza [0,1]
    image = image[..., :3]  # garante 3 canais
    return image

real_labels = []
torch_inputs = []

print("Processando imagens...")
for img, label in dataset:
    img_proc = preprocess_for_inference(img).numpy()
    # A entrada do PyTorch geralmente é [Canal, Altura, Largura]
    # O TensorFlow gera [Altura, Largura, Canal], então precisamos transpor
    tensor = torch.tensor(img_proc, dtype=torch.float32).permute(2, 0, 1)
    torch_inputs.append(tensor)
    real_labels.append(int(label.numpy()))  # salva a classe real (0=Gato, 1=Cachorro)

# --- 2. Empilhar num batch ---
batch_input = torch.stack(torch_inputs, dim=0)  # [100, 3, 224, 224]

print("Batch input shape:", batch_input.shape)

# --- 3. Executar no modelo quantizado ---
# Esta parte é um placeholder, substitua com sua execução real
# executor = TorchExecutor(graph=quanted_graph, device="cpu")
# batch_output = executor(batch_input)

# Para fins de demonstração, vamos simular uma saída
# A saída do seu modelo deve ter o formato [batch_size, 1]
# com as probabilidades de ser a classe 1 (Cachorro)
torch.manual_seed(42) # para resultados reproduzíveis
simulated_logits = torch.randn(100, 1)
batch_output = [torch.sigmoid(simulated_logits)] # Simula a saída de probabilidade

print("\nSaída bruta do modelo (primeiras 5):", batch_output[0][:5].T)
print("Rótulos reais (primeiros 5):", real_labels[:5])


# --- 4. Avaliar os resultados ---
print("\n" + "="*50)
print("INICIANDO AVALIAÇÃO DE DESEMPENHO")
print("="*50)

# O output pode estar em uma lista, então pegamos o primeiro tensor
predictions_tensor = batch_output[0]

# Definimos o limiar de decisão
threshold = 0.5

# Convertendo probabilidades em predições de classe (0 ou 1)
# Se a probabilidade for >= 0.5, prevê 1 (Cachorro), senão 0 (Gato)
predicted_labels_tensor = (predictions_tensor >= threshold).int()

# Convertendo para uma lista simples para facilitar a comparação
predicted_labels = predicted_labels_tensor.squeeze().tolist()

# Contadores para as métricas
correct_predictions = 0
total_samples = len(real_labels)

print("\n--- Análise Detalhada ---")
print(f"{'Índice':<8} | {'Probabilidade':<15} | {'Predição':<10} | {'Real':<8} | {'Resultado'}")
print("-" * 65)

# Iterar sobre cada resultado para comparar
for i in range(total_samples):
    prob = predictions_tensor[i].item()
    pred_label = predicted_labels[i]
    real_label = real_labels[i]

    if pred_label == real_label:
        status = "✅ Acerto"
        correct_predictions += 1
    else:
        status = "❌ Erro"

    print(f"{i:<8} | {prob:<15.4f} | {pred_label:<10} | {real_label:<8} | {status}")

print("-" * 65)

# --- 5. Calcular e exibir a acurácia final ---
accuracy = (correct_predictions / total_samples) * 100

print("\n--- Resumo Final ---")
print(f"Total de Amostras Avaliadas: {total_samples}")
print(f"Total de Acertos: {correct_predictions}")
print(f"Total de Erros: {total_samples - correct_predictions}")
print(f"Acurácia do Modelo: {accuracy:.2f}%")